#Project


#Code to load the datasets

In [ ]:
# Data setup: download datasets from shared Google Drive

!pip -q install gdown

import os
from pathlib import Path

DATA_DIR = Path("/content/ML_Exam_Project")
DATA_DIR.mkdir(exist_ok=True)

FOLDER_URL = "https://drive.google.com/drive/folders/1f8t9t6jLE6vFpnu7A-xQuhieDlhk48Iw?usp=share_link"

!gdown --folder "$FOLDER_URL" -O "$DATA_DIR"

Retrieving folder contents
Processing file 1uSTcAolrashExIz6RzS7xMTdiZY7xsxF test_id.jsonl
Processing file 1-y5suAJGJK-OgY7yNm9LkiWnFmCV0zj7 test_long.jsonl
Processing file 1Gr1newDY91vYEQjdyzshR5LYSU_VqT17 test_ood.jsonl
Processing file 184ksDE_EtHRYNt2FaP4Qvhw85OA1AAR4 train.jsonl
Processing file 1nRoACGPmX7BT9vlG_65aenEiXDJmNlwj validation.jsonl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1uSTcAolrashExIz6RzS7xMTdiZY7xsxF
To: /content/ML_Exam_Project/test_id.jsonl
100% 222k/222k [00:00<00:00, 27.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-y5suAJGJK-OgY7yNm9LkiWnFmCV0zj7
To: /content/ML_Exam_Project/test_long.jsonl
100% 182k/182k [00:00<00:00, 83.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Gr1newDY91vYEQjdyzshR5LYSU_VqT17
To: /content/ML_Exam_Project/test_ood.jsonl
100% 230k/230k [00:00<00:00, 98.0MB/s]
Downloading...
From: https://drive.go

In [ ]:
# Check

from pathlib import Path

DATA_DIR = Path("/content/ML_Exam_Project")

expected_files = [
    "train.jsonl",
    "validation.jsonl",
    "test_id.jsonl",
    "test_ood.jsonl",
    "test_long.jsonl",
]

for filename in expected_files:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"

print("All dataset files found:")
for filename in expected_files:
    print(" -", DATA_DIR / filename)

All dataset files found:
 - /content/ML_Exam_Project/train.jsonl
 - /content/ML_Exam_Project/validation.jsonl
 - /content/ML_Exam_Project/test_id.jsonl
 - /content/ML_Exam_Project/test_ood.jsonl
 - /content/ML_Exam_Project/test_long.jsonl


In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

In [ ]:
# Load the datasets

train_df = load_jsonl(DATA_DIR / "train.jsonl")
validation_df = load_jsonl(DATA_DIR / "validation.jsonl")
test_id_df = load_jsonl(DATA_DIR / "test_id.jsonl")
test_ood_df = load_jsonl(DATA_DIR / "test_ood.jsonl")
test_long_df = load_jsonl(DATA_DIR / "test_long.jsonl")

x_train = train_df["expression"]
y_train = train_df["value"]
x_val = validation_df["expression"]
y_val = validation_df["value"]

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test ID:", test_id_df.shape)
print("Test OOD:", test_ood_df.shape)
print("Test long:", test_long_df.shape)

train_df.head()

Train: (12000, 6)
Validation: (2000, 6)
Test ID: (2000, 6)
Test OOD: (2000, 6)
Test long: (1500, 6)


,id,expression,value,length,operator_count,depth
0,train-00000,4+4+7+3+8-1,25,11,5,5
1,train-00001,5-(6-5)+8+8,20,11,4,5
2,train-00002,5-3+1,3,5,2,3
3,train-00003,1-(1+5+8+7+1),-21,13,5,6
4,train-00004,7+1-(6+5+9),-12,11,4,4


### Output Normalization
####For regression

Normalizing the target variable (output) can help improve model performance and stability during training, especially with regression tasks. By scaling the target values, the model can learn more efficiently. We will normalize `y_train` and `y_val` using the mean and standard deviation calculated from `y_train`.

In [ ]:
# Calculate mean and standard deviation from training targets
y_mean = y_train.mean()
y_std = y_train.std()

print(f"Training target mean: {y_mean:.2f}")
print(f"Training target standard deviation: {y_std:.2f}")

Training target mean: 4.86
Training target standard deviation: 10.59


In [ ]:
# Normalize training and validation targets
y_train_normalized = (y_train - y_mean) / y_std
y_val_normalized = (y_val - y_mean) / y_std

print("Normalized y_train head with corresponding inputs:")
display(pd.DataFrame({'expression': x_train, 'normalized_value': y_train_normalized}).head())
print("Normalized y_val head:")
display(pd.DataFrame({'expression': x_val, 'normalized_value': y_val_normalized}).head())

Normalized y_train head with corresponding inputs:


,expression,normalized_value
0,4+4+7+3+8-1,1.901590
1,5-(6-5)+8+8,1.429479
2,5-3+1,-0.175696
3,1-(1+5+8+7+1),-2.441825
4,7+1-(6+5+9),-1.592027


Normalized y_val head:


,expression,normalized_value
0,4-(1-7),0.485259
1,6+6-9,-0.175696
2,0+0+8,0.296414
3,8-(3-7),0.674103
4,4-0-(8+1)-(4-1),-1.214339


##Tokenization

In [ ]:
import random
import math
import copy
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
all_chars = set()                   #Counting all the characters in every training expression and adding it to all_chars if new
for expression in x_train:
    for char in expression:
        all_chars.add(char)


V = sorted(list(all_chars))
PAD_TOKEN = "[PAD]"
V = [PAD_TOKEN] + V
print(len(V))
token_to_index = {token: i for i, token in enumerate(V)}
print(token_to_index)

15
{'[PAD]': 0, '(': 1, ')': 2, '+': 3, '-': 4, '0': 5, '1': 6, '2': 7, '3': 8, '4': 9, '5': 10, '6': 11, '7': 12, '8': 13, '9': 14}


##CNN

In [ ]:
def encode(x):                                                        #to see if the one hot encoding/decoding is functioning well
    return [token_to_index[token] for token in list(x)]

def decode(x):
    return ''.join([V[index] for index in x if V[index] != PAD_TOKEN])

print(encode('+'))
print(decode(encode('+')))

[3]
+


In [ ]:
def collate_fn(batch):                #we dynamically pad every batch that has to be given to the nn by the length of the maximum element in it
    # 'batch' is a list of (input_sequence, target_value) tuples
    # For us, input_sequence is x_data, and target_value is y_data

    # Separate inputs (expressions) and targets (values)
    expressions = [item[0] for item in batch]
    values = [item[1] for item in batch]

    # Encode expressions
    encoded_expressions = [encode(expr) for expr in expressions]

    # Find the maximum length in the current batch
    batch_max_length = max(len(seq) for seq in encoded_expressions)

    # Pad each sequence in the batch to batch_max_length
    padded_expressions = []
    for seq in encoded_expressions:
        padding_needed = batch_max_length - len(seq)
        padded_seq = seq + [token_to_index[PAD_TOKEN]] * padding_needed
        padded_expressions.append(padded_seq)

    # Convert to PyTorch tensors
    x_batch = torch.tensor(padded_expressions, dtype=torch.long)
    y_batch = torch.tensor(values, dtype=torch.float)

    return x_batch, y_batch

In [ ]:
VOCAB_SIZE = len(V)
#EMBEDDING_DIM = 64 # Dimension for token embeddings
NUM_OUTPUTS = 1 # For regression, we predict a single value (the result of the expression)

#THIS IS GIVEN BY CHATGPT

#Another way to reason is by using an embedding dimension, transforming all the input data to a fixed series of vectors, given that we want to try one hot encoding this is another apporach

import torch.nn as nn
import torch.nn.functional as F

class ExpressionCNNOneHot(nn.Module):                               #we are defining a new model to enable the dynamic padding of the inputs
    def __init__(self, vocab_size, hidden_channels=64, num_outputs=1):            #num_outputs is for classification against regression problems, 199 for classification, 1 for regression
        super(ExpressionCNNOneHot, self).__init__()                         #all the attributes of the nn.Module class, (we need to see it a little bit)

        self.conv1 = nn.Conv1d(                                         #in this type of model we add 2 convolution, both of size 3, the inputs of the first one have the vocab size as dimension, the outputs have dimension hidden_channels
            in_channels=vocab_size,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1                                                     #CONVOLUTIONAL padding
        )

        self.conv2 = nn.Conv1d(                                           #the inputs and the outputs of the of the second one have both the size of hidden_channels, we can change this
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1
        )
#we can change the structure by make layers of different sizes, not only fixing one hidden dimension
        self.fc = nn.Linear(hidden_channels, num_outputs)                       #output function, if we use classification we have also to use softmax or directly argmax in the training (We have to check it)

    def forward(self, x, mask):
        # x shape: (batch_size, vocab_size, sequence_length)
        # mask shape: (batch_size, sequence_length)

#we should had a normalization of the parameters
        h = F.relu(self.conv1(x))                                                 #first layer with sigma = ReLU
        h = F.relu(self.conv2(h))                                                   #Second layer with sigma = ReLU, h is the output

# Masked mean pooling
# h shape: (batch_size, hidden_channels, sequence_length)

        #after applying the neural network, we need a way to obtain one vector instead of the whole expression, therefore masked mean pooling (mean pooling but we don't count PAD tokens)
        #mask will be a tensor of the same shame as the input sequence, an element in this tensor can be 1, if the corresponding term of the sequence is not PAD, and 0, if it is PAD

        mask = mask.unsqueeze(1)  # (batch_size, 1, sequence_length)                #to match the output, we have to add antoher dimension corresponding to the hidden channels of h, before this mask has shape (batch_size, max_sequence_length), after this it has shape (batch_size, 1, sequence length). The PyTorch's broadcasting rule comes in action in the multiplication letting this added dimension expand to the size of hidden_channels.
        h = h * mask            #we multiply element wise the output h by mask, possible becuase of the unsqueezing. in the multiplication, every PAD token will be multiplied by zero for every hidden channel

        lengths = mask.sum(dim=2).clamp(min=1)  # (batch_size, 1). mask.sum(dim=2) sums all the elements of the last dimension (sequence_length): given that we applied already multiplied by mas this will give us the actual length of the inputs. clamp(min = 1) is for avoiding lengths being zero (if they are only made by PAD tokens)

        pooled = h.sum(dim=2) / lengths  # (batch_size, hidden_channels). it sums all the characters in every elements of (batch_size, hidden_channels) (i. e. every expression in every hidden channel) and divides by the real length (masked mean pooling).

        output = self.fc(pooled).squeeze(1)       #with the pooled output we apply the final linear transfomration, squeezing at the end the hidden_channels part therefore obtaining a vector of size = batch_size in which every number is the prediction of the value of the expression.

        return output

In [ ]:
#creation of the model


model = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

###Training the model

In [ ]:
def train(model, x_train_data, y_train_data, optimizer, loss_fn, epochs, device, batch_size, x_val_data=None, y_val_data=None):
    model.to(device)
    # Create DataLoader for training data
    train_dataset = list(zip(x_train_data, y_train_data))             #we don't use train_df because it has more than 2 elements, it wouldn' know which are the inputs and which are the outputs. given that we already splitted the train_df dataset, it is easier to concatenate these two elements
                                                                    #zip paires every i-th element of x_train_data with the i-th element of y_train_data. list creates a list containing as every elements tuples of (input, output)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    # Create DataLoader for validation data if provided
    val_loader = None
    if x_val_data is not None and y_val_data is not None:
        val_dataset = list(zip(x_val_data, y_val_data))
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)        #collate_fn does the padding of every element in a batch

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch_idx, (x_batch, y_batch) in enumerate(train_loader):               #enumerate is useful to enumerate every batch
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()             #the mask useful for the forward method in the definition of the class.

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)      #the one_hot sends every token in every expression of the x_batch in a vectore inside R^(VOCAB_SIZE), float converts every element into a vector of floats.
                                                                                                #permute(0, 2, 1) is because the convolutional layers usually want an input of shapes (batch_size, VOCAB_SIZE, sequence_length), right now it is (batch_size, sequence_length, VOCAB_SIZE)
            optimizer.zero_grad()
            y_pred = model(x_batch, mask)       #in nn.Modules, whenever you call model() you are actually doing model.forward(), therefore this would actually be model.forward()
            loss = loss_fn(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = np.mean(losses)
        print(f'Train Epoch: {epoch + 1}, Train Loss = {train_loss:.4f}')

        if val_loader is not None:
            val_loss = evaluate(model, val_loader, loss_fn, device)
            print(f'Validation Loss: {val_loss:.4f}')

def evaluate(model, loader, loss_fn, device):
    model.to(device)
    model.eval()
    total_loss = 0
    num_samples = 0
    # The 'correct' variable is not used in a regression context, so it will be removed.
    # correct = 0 # No longer relevant for regression metrics

    with torch.no_grad():
        for batch_size, (x_batch, y_batch) in enumerate(loader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)

            y_pred = model(x_batch, mask)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * x_batch.size(0)         #we add to the total loss the average loss in the x_batch as many times as the elements in x_batch (remember x_batch = (batch_size, VOCAB_SIZE, sequence_length))
            num_samples += x_batch.size(0)
            # The following line is for classification accuracy and is not relevant for regression.
            # correct += y_pred.eq(y_batch.view_as(y_pred)).sum().item()

    # The 'accuracy' calculation is for classification and will be removed.
    # accuracy = correct/num_samples
    average_loss = total_loss / num_samples
    # Updated print statement for regression, removing accuracy.
    print(f'Validation set: Average loss: {average_loss:.4f}')
    return average_loss

In [ ]:
batch_size = 32
learning_rate = 0.001
epochs = 40
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
loss_fn = nn.MSELoss() # Changed from CrossEntropyLoss to MSELoss for regression
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train(model, x_train, y_train, optimizer, loss_fn, epochs, device, batch_size, x_val, y_val)          #it is normal for regression, we should normalize the datas and the rescale them, the important thing is that it is decreasing

Train Epoch: 1, Train Loss = 83.4417
Validation set: Average loss: 59.2983
Validation Loss: 59.2983
Train Epoch: 2, Train Loss = 44.8149
Validation set: Average loss: 31.0153
Validation Loss: 31.0153
Train Epoch: 3, Train Loss = 27.8490
Validation set: Average loss: 25.5625
Validation Loss: 25.5625
Train Epoch: 4, Train Loss = 23.4826
Validation set: Average loss: 22.1001
Validation Loss: 22.1001
Train Epoch: 5, Train Loss = 20.7980
Validation set: Average loss: 21.1745
Validation Loss: 21.1745
Train Epoch: 6, Train Loss = 18.7260
Validation set: Average loss: 17.6648
Validation Loss: 17.6648
Train Epoch: 7, Train Loss = 16.9716
Validation set: Average loss: 16.5171
Validation Loss: 16.5171
Train Epoch: 8, Train Loss = 15.8683
Validation set: Average loss: 16.7665
Validation Loss: 16.7665
Train Epoch: 9, Train Loss = 15.0085
Validation set: Average loss: 14.5479
Validation Loss: 14.5479
Train Epoch: 10, Train Loss = 14.1894
Validation set: Average loss: 13.8980
Validation Loss: 13.8980

In [ ]:
# Re-run training with normalized outputs



model_normal = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

# Reset optimizer as model has been re-instantiated
optimizer_normal = torch.optim.Adam(model_normal.parameters(), lr=learning_rate)

print("\n--- Training with Normalized Outputs ---")
train(model_normal, x_train, y_train_normalized, optimizer_normal, loss_fn, epochs, device, batch_size, x_val, y_val_normalized)


--- Training with Normalized Outputs ---
Train Epoch: 1, Train Loss = 0.4681
Validation set: Average loss: 0.2252
Validation Loss: 0.2252
Train Epoch: 2, Train Loss = 0.1902
Validation set: Average loss: 0.1608
Validation Loss: 0.1608
Train Epoch: 3, Train Loss = 0.1543
Validation set: Average loss: 0.1383
Validation Loss: 0.1383
Train Epoch: 4, Train Loss = 0.1387
Validation set: Average loss: 0.1257
Validation Loss: 0.1257
Train Epoch: 5, Train Loss = 0.1218
Validation set: Average loss: 0.1275
Validation Loss: 0.1275
Train Epoch: 6, Train Loss = 0.1106
Validation set: Average loss: 0.1077
Validation Loss: 0.1077
Train Epoch: 7, Train Loss = 0.1029
Validation set: Average loss: 0.1206
Validation Loss: 0.1206
Train Epoch: 8, Train Loss = 0.0995
Validation set: Average loss: 0.0986
Validation Loss: 0.0986
Train Epoch: 9, Train Loss = 0.0933
Validation set: Average loss: 0.0965
Validation Loss: 0.0965
Train Epoch: 10, Train Loss = 0.0882
Validation set: Average loss: 0.0966
Validation 

By normalizing the outputs, the model should now be trained on a scaled version of the target values. This often leads to a more stable training process and potentially better convergence. The loss values reported are now for the normalized predictions. To get the actual predicted values, you would need to denormalize the model's output using `(prediction * y_std) + y_mean`. Given that the output usually is an integer we should also transform the data back into an integer